# Expected Value in Poker

**Expected Value (EV)** is the single most important concept in poker strategy. Every decision — call, fold, bet, raise — can be evaluated as a probability-weighted average of all possible outcomes.

$$\text{EV} = \sum_{i} P(\text{outcome}_i) \times \text{Value}(\text{outcome}_i)$$

A **+EV** play wins money on average over many repetitions. A **-EV** play loses money. Good poker is about consistently making +EV decisions regardless of short-term results.

This notebook builds EV intuition from first principles, then applies it to real hand scenarios using the `pkpy` library.

In [1]:
# Standard library
from itertools import permutations

# Third-party
import pkpy

## 1. EV Fundamentals

Before applying EV to poker hands, let's build intuition with a simple example: a coin-flip bet.

You are offered a bet where:
- You win **$3** if the coin lands heads (50%)
- You lose **$2** if the coin lands tails (50%)

Should you take this bet?

In [2]:
def ev(outcomes: list[tuple[float, float]]) -> float:
    """Compute expected value from a list of (probability, value) pairs."""
    return sum(prob * value for prob, value in outcomes)


coin_flip_ev = ev([
    (0.50, +3.0),   # heads: win $3
    (0.50, -2.0),   # tails: lose $2
])

print(f"Coin-flip EV: ${coin_flip_ev:+.2f} per bet")
print(f"Decision: {'Take it (+EV)' if coin_flip_ev > 0 else 'Decline (-EV)'}")

Coin-flip EV: $+0.50 per bet
Decision: Take it (+EV)


The EV formula generalises directly to poker. In poker, "outcomes" are: win the pot, lose your investment, or chop the pot in a tie. Probabilities come from hand equities.

### Quiz 1 — EV Fundamentals

A game offers two outcomes:
- Win **$6** with **35%** probability
- Lose **$2** with **65%** probability

**Q1:** What is the EV of one play? Enter your answer in dollars (e.g. `0.80`).

**Q2:** A casino offers a "special" bet: win $10 (10% chance), win $1 (40% chance), lose $3 (50% chance). What's the EV? Enter in dollars.

In [3]:
from jupyterquiz import display_quiz

quiz_1 = [
    {
        "question": "A game pays +6(352 (65% chance). What is the EV in dollars?",
        "type": "numeric",
        "precision": 2,
        "answers": [
            {
                "type": "value",
                "value": 0.80,
                "correct": True,
                "feedback": "Correct! EV = 0.35×6+0.65×(−2) = 2.10−1.30 = +$0.80. Take this bet every time.",
            },
            {
                "type": "range",
                "range": [0.79, 0.81],
                "correct": True,
                "feedback": "Correct! EV = 0.35×6+0.65×(−2) = +$0.80.",
            },
        ],
    },
    {
        "question": "Casino special bet: win 10(101 (40%), lose $3 (50%). What is the EV in dollars?",
        "type": "numeric",
        "precision": 2,
        "answers": [
            {
                "type": "value",
                "value": -0.10,
                "correct": True,
                "feedback": "Correct! EV = 0.10×10+0.40×1 + 0.50×(−3)=1.00 + 0.40−1.50 = −$0.10. Skip it.",
            },
            {
                "type": "range",
                "range": [-0.11, -0.09],
                "correct": True,
                "feedback": "Correct! EV = −$0.10. The big win looks tempting but it's too rare.",
            },
        ],
    },
]

display_quiz(quiz_1)

<IPython.core.display.Javascript object>

## 2. Pot Odds & Break-Even Equity

**Pot odds** are the ratio of the pot size to the cost of calling. They tell you the minimum equity (win probability) you need for a call to be +EV.

When facing a bet:

$$\text{Break-even equity} = \frac{\text{Call Amount}}{\text{Pot after call}}$$

If your actual equity **≥** break-even equity → call is +EV.  
If your actual equity **<** break-even equity → fold is +EV.

In [ ]:
def pot_odds_call_ev(
    pot: float,
    bet: float,
    equity: float,
) -> float:
    """
    EV of calling a bet.

    Args:
        pot:    Chips already in pot before the bet.
        bet:    Size of the bet we face.
        equity: Our probability of winning (0-1).

    Returns:
        EV of the call relative to folding (0 = indifferent).
    """
    total_pot = pot + bet + bet  # pot + villain bet + our call
    ev_win = equity * total_pot
    ev_lose = (1 - equity) * (-bet)
    return ev_win + ev_lose


def break_even_equity(pot: float, bet: float) -> float:
    """Minimum equity for a call to be +EV."""
    return bet / (pot + bet + bet)


# Scenario: 100pot,villainbets50, we have 35% equity (e.g. a flush draw)
scenarios = [
    {"label": "Flush draw vs pot-sized bet",  "pot": 100, "bet": 100, "equity": 0.36},
    {"label": "Flush draw vs half-pot bet",   "pot": 100, "bet":  50, "equity": 0.36},
    {"label": "Overpair vs all-in shove",     "pot":  80, "bet": 200, "equity": 0.72},
    {"label": "Bottom pair vs small bet",     "pot": 100, "bet":  25, "equity": 0.22},
]

print(f"{'Scenario':<38} {'Pot':>6} {'Bet':>6} {'Equity':>8} {'Break-even':>12} {'Call EV':>10} {'Decision':>10}")
print("-" * 100)
for s in scenarios:
    be = break_even_equity(s["pot"], s["bet"])
    call_ev = pot_odds_call_ev(s["pot"], s["bet"], s["equity"])
    decision = "+EV Call" if call_ev > 0 else "-EV Fold"
    print(
        f"{s['label']:<38} "
        f"${s['pot']:>5} "
        f"${s['bet']:>5} "
        f"{s['equity']:>7.1%} "
        f"{be:>11.1%} "
        f"${call_ev:>+9.2f} "
        f"{decision:>10}"
    )

### Quiz 2 — Pot Odds & Break-Even Equity

You face a bet in a live game:
- Pot before villain's bet: **$150**
- Villain bets: **$75**
- Your estimated equity: **28%**

**Q1:** What is the break-even equity for this call? Enter as a percentage (e.g. `25`).

**Q2:** Is this call +EV or −EV?

**Q3:** Your read changes — villain is actually on a draw and you now put them on 22% equity for villain (meaning you have **78%** equity). Does calling become even more +EV, or is your equity already well above the threshold that it doesn't matter?

In [ ]:
quiz_2 = [
    {
        "question": "Pot: 150,villainbets75. What is the break-even equity (%)? Enter as a whole number.",
        "type": "numeric",
        "precision": 0,
        "answers": [
            {
                "type": "value",
                "value": 25,
                "correct": True,
                "feedback": "Correct! Break-even = 75 / (150 + 75 + 75) = 75/300 = 25%. You need at least 25% equity to call.",
            },
        ],
    },
    {
        "question": "You have 28% equity vs the 25% break-even threshold. What is the correct action?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "Call — it is +EV",
                "correct": True,
                "feedback": "Correct! EV(call) = 0.28×300−0.72×75 = 84−54 = +$30. Your 28% exceeds the 25% threshold.",
            },
            {
                "answer": "Fold — the pot odds aren't good enough",
                "correct": False,
                "feedback": "Incorrect. Break-even equity is 25%; you have 28%, so calling is +EV by $30.",
            },
            {
                "answer": "It depends on position",
                "correct": False,
                "feedback": "Position matters for many decisions, but pure pot-odds calls are decided by equity vs break-even equity alone.",
            },
        ],
    },
    {
        "question": "Your equity is well above break-even (28% vs 25%). If your equity rises to 78%, which statement is true?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "The call becomes even more +EV — higher equity always means higher EV",
                "correct": True,
                "feedback": "Correct! EV(call) = 0.78×$300 − 0.22×$75 = $234 − $16.50 = +$217.50. More equity = more EV.",
            },
            {
                "answer": "EV stays the same once you're above break-even",
                "correct": False,
                "feedback": "No — EV is linear in equity. Every percentage point above break-even adds more expected value.",
            },
            {
                "answer": "You should raise instead, so EV of calling is irrelevant",
                "correct": False,
                "feedback": "Raising may also be correct, but the EV of calling still increases with equity — the two aren't mutually exclusive to calculate.",
            },
        ],
    },
]

display_quiz(quiz_2)

## 3. Preflop All-In EV vs. a Range

In a preflop all-in spot, EV is straightforward: you are either called and face a showdown, or your shove folds them out.

Here we model calling an all-in shove. The villain has shoved, you've been given a range for their shoving range. We use `pkpy.Versus` to compute equity against each specific hand in their range, then calculate the EV of calling vs folding.

**EV(call)** = equity × total_pot - (1 - equity) × call_amount  
**EV(fold)** = 0 (by convention — relative to current stack)

In [ ]:
def preflop_allin_ev(
    hero_hand: str,
    villain_range: str,
    effective_stack: float,
    pot_before_call: float,
) -> dict:
    """
    EV of calling a preflop all-in shove.

    Args:
        hero_hand:        Two-card hand string, e.g. "Ks Kh".
        villain_range:    Combo range string, e.g. "QQ+,AKs,AKo".
        effective_stack:  Chips at risk (the all-in amount).
        pot_before_call:  Chips already in the pot (blinds, dead money, etc.)
                          before hero's call completes it.

    Returns:
        Dict with equity, ev_call, ev_fold, and recommendation.
    """
    hero = pkpy.Two.parse(hero_hand)
    villain = pkpy.Combos.parse(villain_range)

    solver = pkpy.Versus(hero, villain)
    hups = solver.hups_at_deal()
    combined = pkpy.Versus.combined_odds_at_deal(hups)

    equity = combined.win_percentage() / 100.0
    # Half credit for ties
    equity += (combined.draw_percentage() / 100.0) * 0.5

    total_pot = pot_before_call + effective_stack
    ev_call = equity * total_pot - (1 - equity) * effective_stack

    return {
        "hero": hero_hand,
        "range": villain_range,
        "equity": equity,
        "ev_call": ev_call,
        "ev_fold": 0.0,
        "recommendation": "CALL" if ev_call > 0 else "FOLD",
        "combo_count": len(hups),
    }


# Classic preflop spots: 100BB effective, 3BB already in pot (blinds)
spots = [
    ("Ks Kh", "AA",           97, 3),   # KK vs AA only — dominated
    ("Ks Kh", "QQ+, AKs, AKo", 97, 3),  # KK vs a wider range
    ("Ac Kd", "QQ+, AKs, AKo", 97, 3),  # AKo vs the same range
    ("Qh Qd", "KK+, AKs, AKo", 97, 3),  # QQ facing a tighter 4-bet range
]

print(f"{'Hero':<8} {'Range':<18} {'Combos':>7} {'Equity':>8} {'EV(call)':>10} {'Action':>8}")
print("-" * 65)
for hand, rng, stack, pot in spots:
    result = preflop_allin_ev(hand, rng, stack, pot)
    print(
        f"{result['hero']:<8} "
        f"{result['range']:<18} "
        f"{result['combo_count']:>7} "
        f"{result['equity']:>7.1%} "
        f"${result['ev_call']:>+9.1f} "
        f"{result['recommendation']:>8}"
    )

### Quiz 3 — Preflop All-In EV

Review the all-in results printed above, then answer:

**Q1:** KK vs an AA-only range — KK wins roughly 18%. With 3BB already in the pot and 97BB effective stacks (197BB total pot), what is the approximate EV of calling? Enter in chips (e.g. `-44.1`).

**Q2:** Why does AKo have *positive* EV calling all-in vs the range `QQ+, AKs, AKo`?

**Q3:** True or False — adding JJ to villain's shoving range always increases hero's EV when calling with AKo.

In [ ]:
quiz_3 = [
    {
        "question": (
            "KK vs AA-only. KK has ~18% equity. "
            "Pot before call: 3BB, effective stack: 97BB (total pot after call: 197BB). "
            "What is the approximate EV of calling (in BBs)? Enter to 1 decimal place."
        ),
        "type": "numeric",
        "precision": 1,
        "answers": [
            {
                "type": "range",
                "range": [-45.0, -43.0],
                "correct": True,
                "feedback": (
                    "Correct! EV = 0.18×197 − 0.82×97 = 35.46 − 79.54 = −44.1 BBs. "
                    "Even with dead money, KK is such a big underdog to AA that calling is −EV."
                ),
            },
        ],
    },
    {
        "question": "Why does AKo have positive EV calling all-in vs the range QQ+, AKs, AKo?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "(a) Because AK beats QQ at showdown",
                "correct": False,
                "feedback": "AK does beat QQ at showdown — but that's not why the EV is positive. The equity distribution across the whole range matters.",
            },
            {
                "answer": "(b) Because QQ makes up the majority of the range, and AK has ~46% equity vs QQ",
                "correct": True,
                "feedback": "Correct! QQ has 6 combos (the most in the range). AK is ~46% vs QQ, ~50% vs AKs/AKo, and a big dog to KK/AA. The QQ combos pull the aggregate equity above break-even.",
            },
            {
                "answer": "(c) Because AK blocks AA and KK via card removal",
                "correct": False,
                "feedback": "Blockers do reduce the number of AA/KK combos, which helps — but they're not the primary reason EV is positive here.",
            },
            {
                "answer": "(d) Because AKo is always a preflop favourite",
                "correct": False,
                "feedback": "AKo is NOT always a favourite preflop — it is a dog to any pair QQ and above.",
            },
        ],
    },
    {
        "question": "True or False: Adding JJ to villain's shoving range always increases hero's EV when calling with AKo.",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "True",
                "correct": False,
                "feedback": "False! JJ is a ~56% favourite over AKo. Adding more combos that are *ahead* of your hand hurts your equity and lowers your EV.",
            },
            {
                "answer": "False",
                "correct": True,
                "feedback": "Correct! JJ beats AKo ~56% of the time. Adding JJ combos to the range reduces hero's aggregate equity — it does NOT help.",
            },
        ],
    },
]

display_quiz(quiz_3)

## 4. Drawing Hand EV on the Turn

With one card to come (the river), we can use `pkpy.Game.turn_case_evals()` to enumerate all 44 possible river cards and see exactly which ones we win and lose with. This gives us a precise equity calculation to plug into the EV formula.

Classic spot: flush draw facing a bet on the turn.

In [ ]:
def draw_ev_on_turn(
    hero_hand: str,
    villain_hand: str,
    board_str: str,
    pot: float,
    villain_bet: float,
) -> dict:
    """
    EV of calling a turn bet with one card to come.

    Enumerates all river runouts to compute exact equity.

    Args:
        hero_hand:   Hero's two hole cards, e.g. "Ah Th".
        villain_hand: Villain's two hole cards, e.g. "Kd Ks".
        board_str:   Turn board (4 cards), e.g. "9h 7h 2c 5d".
        pot:         Pot size before villain's bet.
        villain_bet: Size of villain's bet.

    Returns:
        Dict with equity, outs, ev_call, ev_fold, recommendation.
    """
    hc = pkpy.HoleCards.parse(f"{hero_hand} {villain_hand}")
    board = pkpy.Board.parse(board_str)
    game = pkpy.Game(hc, board)

    case_evals = game.turn_case_evals()
    outs_obj = pkpy.Outs.from_case_evals(case_evals)

    total_rivers = len(case_evals)
    hero_outs = outs_obj.len_for_player(1)   # player index 1 = hero (first in HoleCards)
    villain_outs = outs_obj.len_for_player(2)

    equity = hero_outs / total_rivers
    total_pot_if_call = pot + villain_bet + villain_bet
    ev_call = equity * total_pot_if_call - (1 - equity) * villain_bet

    return {
        "hero": hero_hand,
        "villain": villain_hand,
        "board": board_str,
        "total_rivers": total_rivers,
        "hero_outs": hero_outs,
        "villain_outs": villain_outs,
        "equity": equity,
        "ev_call": ev_call,
        "ev_fold": 0.0,
        "recommendation": "CALL" if ev_call > 0 else "FOLD",
    }


# Scenario 1: Nut flush draw on the turn facing a pot-sized bet
# Hero: Ah Th (nut flush draw) vs Villain: Kd Ks (top set)
# Board: 9h 7h 2c 5d (turn)
draw1 = draw_ev_on_turn("Ah Th", "Kd Ks", "9h 7h 2c 5d", pot=100, villain_bet=100)

# Scenario 2: Weak flush draw on the turn facing a half-pot bet
# Hero: 6h 4h vs Villain: Ac Ad
# Board: Kh Th 2c 7d
draw2 = draw_ev_on_turn("6h 4h", "Ac Ad", "Kh Th 2c 7d", pot=100, villain_bet=50)

for draw in [draw1, draw2]:
    be = draw["villain_bet"] / (draw["hero_outs"] * 0 + 100 + draw["villain_bet"] * 2)  # recalc for display
    be = draw_ev_on_turn.__doc__  # skip, print inline
    break_even = draw["villain_bet"] / (100 + draw["villain_bet"] * 2)

print("Scenario 1: Nut Flush Draw vs Pot-Sized Bet")
print(f"  Board       : {draw1['board']}")
print(f"  Hero        : {draw1['hero']}  |  Villain: {draw1['villain']}")
print(f"  River cards : {draw1['total_rivers']} possible")
print(f"  Hero outs   : {draw1['hero_outs']} / {draw1['total_rivers']}")
print(f"  Equity      : {draw1['equity']:.1%}")
print(f"  Break-even  : {draw1['villain_bet'] / (draw1['total_rivers'] * 0 + 100 + draw1['villain_bet'] * 2):.1%}")
print(f"  EV(call)    : ${draw1['ev_call']:+.2f}")
print(f"  Decision    : {draw1['recommendation']}")
print()
print("Scenario 2: Weak Flush Draw vs Half-Pot Bet")
print(f"  Board       : {draw2['board']}")
print(f"  Hero        : {draw2['hero']}  |  Villain: {draw2['villain']}")
print(f"  River cards : {draw2['total_rivers']} possible")
print(f"  Hero outs   : {draw2['hero_outs']} / {draw2['total_rivers']}")
print(f"  Equity      : {draw2['equity']:.1%}")
print(f"  Break-even  : {draw2['villain_bet'] / (100 + draw2['villain_bet'] * 2):.1%}")
print(f"  EV(call)    : ${draw2['ev_call']:+.2f}")
print(f"  Decision    : {draw2['recommendation']}")

### Quiz 4 — Drawing Hand EV on the Turn

**Q1:** A standard flush draw has 9 outs. With 44 remaining cards (turn → river), what is the exact equity?

**Q2:** In Scenario 1 above, why might `hero_outs` for `Ah Th` exceed the 9 you'd expect from a flush draw alone?

**Q3 (Fill in the blank):** If villain bets **half the pot** ($50 into $100), you need ____% equity to break even on a call.

> Run the answer cell below to check.

In [ ]:
# Quiz 4 — Answer Reveal

# Q1: standard 9-out flush draw vs 44 remaining cards
outs_flush = 9
rivers = 44
equity_flush = outs_flush / rivers
print(f"Q1  9 outs / 44 rivers = {equity_flush:.1%}")
print(f"    (Classic approximation: 9 × 2% ≈ 18% — slightly off because there are 44 not 50 cards)")
print()

# Q2: why can hero_outs exceed 9?
print("Q2  pkpy counts ALL river cards where hero wins — not just flush cards.")
print("    Ah Th can win by pairing the ace (if it beats villain's pair rank),")
print("    making a straight, or any other combination giving the best 5-card hand.")
print("    'Outs' in pkpy = 'winning river cards', a superset of named draw outs.")
print()

# Q3: break-even for half-pot bet
pot_q3, bet_q3 = 100, 50
be_q3 = break_even_equity(pot_q3, bet_q3)
print(f"Q3  Break-even = {bet_q3} / ({pot_q3} + 2×{bet_q3}) = {bet_q3}/{pot_q3 + 2*bet_q3} = {be_q3:.1%}")
print(f"    A half-pot bet requires only {be_q3:.0%} equity — a very low calling threshold.")

## 5. Kuhn Poker: Full Game Tree EV

**Kuhn Poker** is a minimal 3-card game invented by Harold Kuhn in 1950. It uses only Jack (J), Queen (Q), and King (K). Each player antes 1 chip, then each receives one card. Player 0 acts first (check or bet 1 chip), then Player 1, with possible back-and-forth. High card wins the showdown.

Because the deck has only 3 cards, we can **enumerate every possible game** exactly — 6 possible deals × all possible action sequences — giving a complete analytical picture of EV.

The `pkpy` library provides `KuhnState` which models this game tree exactly.

In [ ]:
def kuhn_game_tree(p0_card: pkpy.KuhnCard, p1_card: pkpy.KuhnCard) -> list[dict]:
    """
    Enumerate all terminal nodes in the Kuhn Poker game tree for a given deal.

    Returns a list of dicts with action history, payoffs, and the path taken.
    """
    terminal_nodes = []

    def traverse(state: pkpy.KuhnState, path: list[str]):
        if state.is_terminal():
            payoff = state.payoff()
            terminal_nodes.append({
                "path": " → ".join(path) if path else "(no action)",
                "p0_payoff": payoff[0],
                "p1_payoff": payoff[1],
            })
            return
        for action in state.legal_actions():
            traverse(state.apply(action), path + [str(action)])

    root = pkpy.KuhnState(p0_card, p1_card)
    traverse(root, [])
    return terminal_nodes


# Print the full game tree for J vs Q (the most interesting matchup)
cards = [pkpy.KuhnCard.JACK, pkpy.KuhnCard.QUEEN, pkpy.KuhnCard.KING]
card_names = {pkpy.KuhnCard.JACK: "J", pkpy.KuhnCard.QUEEN: "Q", pkpy.KuhnCard.KING: "K"}

print("Complete game tree for P0=Jack, P1=Queen")
print(f"{'Action sequence':<30} {'P0 payoff':>10} {'P1 payoff':>10}")
print("-" * 54)
for node in kuhn_game_tree(pkpy.KuhnCard.JACK, pkpy.KuhnCard.QUEEN):
    print(f"{node['path']:<30} {node['p0_payoff']:>+10} {node['p1_payoff']:>+10}")

### Quiz 5 — Kuhn Poker Game Tree

Study the game tree for P0=Jack, P1=Queen printed above, then answer:

**Q1:** How many terminal nodes exist for the J vs Q deal? (Count them from the output.)

**Q2:** In the path `Check → Bet → Fold`, *who* folds and what is P0's payoff?

**Q3 (True/False):** The number of terminal nodes is the same for every possible deal (J vs Q, J vs K, Q vs K, etc.).

> Run the answer cell below to check.

In [ ]:
# Quiz 5 — Answer Reveal

# Q1: count terminal nodes for J vs Q
jq_nodes = kuhn_game_tree(pkpy.KuhnCard.JACK, pkpy.KuhnCard.QUEEN)
print(f"Q1  Terminal nodes for J vs Q: {len(jq_nodes)}")
for node in jq_nodes:
    print(f"    {node['path']:<30} P0: {node['p0_payoff']:+}  P1: {node['p1_payoff']:+}")
print()

# Q2: Check → Bet → Fold
print("Q2  'Check → Bet → Fold':")
print("    P0 checks, P1 bets 1 chip, P0 FOLDS.")
print("    P0 only invested the 1-chip ante → payoff is −1.")
print("    P1 collects the ante without a showdown → payoff is +1.")
print()

# Q3: same structure for every deal?
all_counts = {}
for c0 in [pkpy.KuhnCard.JACK, pkpy.KuhnCard.QUEEN, pkpy.KuhnCard.KING]:
    for c1 in [pkpy.KuhnCard.JACK, pkpy.KuhnCard.QUEEN, pkpy.KuhnCard.KING]:
        if c0 != c1:
            nodes = kuhn_game_tree(c0, c1)
            all_counts[f"{card_names[c0]} vs {card_names[c1]}"] = len(nodes)

print("Q3  Terminal node counts per deal:")
for deal, count in all_counts.items():
    print(f"    {deal}: {count}")
unique = set(all_counts.values())
verdict = "TRUE" if len(unique) == 1 else "FALSE"
print(f"    → {verdict} — the game TREE structure is identical for every deal;")
print(f"       only the payoffs at the leaves differ based on card ranks.")

## 6. EV Under Different Strategies

Now let's compute the **expected value per hand** for two strategies:

1. **Default strategy** — a naive heuristic (always check with weak hands, always bet with strong)
2. **GTO strategy** — the Nash equilibrium solution parametrised by `alpha` (P0's Jack bet frequency)

The Nash equilibrium EV for Player 0 in Kuhn Poker is **-1/18 ≈ -0.0556** chips per hand. Player 0 is at a structural disadvantage because they act first without information.

In [ ]:
def compute_kuhn_ev(strategy: pkpy.KuhnStrategy) -> tuple[float, float]:
    """
    Compute Player 0 and Player 1 EV per hand under a given strategy.

    Averages across all 6 equally-likely deals (permutations of 3 cards
    taken 2 at a time without replacement).

    Args:
        strategy: A KuhnStrategy instance.

    Returns:
        (ev_p0, ev_p1) expected chips per hand.
    """
    cards = [pkpy.KuhnCard.JACK, pkpy.KuhnCard.QUEEN, pkpy.KuhnCard.KING]
    deals = [(c0, c1) for c0 in cards for c1 in cards if c0 != c1]

    def ev_for_deal(
        state: pkpy.KuhnState,
        reach_p0: float = 1.0,
        reach_p1: float = 1.0,
    ) -> tuple[float, float]:
        """Recursive counterfactual reach-weighted EV traversal."""
        if state.is_terminal():
            payoff = state.payoff()
            weight = reach_p0 * reach_p1
            return weight * payoff[0], weight * payoff[1]

        current = state.current_player()
        action_probs = dict(strategy.action_probs(state.info_set(current)))

        total_ev0, total_ev1 = 0.0, 0.0
        for action, prob in action_probs.items():
            if current == 0:
                sub0, sub1 = ev_for_deal(state.apply(action), reach_p0 * prob, reach_p1)
            else:
                sub0, sub1 = ev_for_deal(state.apply(action), reach_p0, reach_p1 * prob)
            total_ev0 += sub0
            total_ev1 += sub1

        return total_ev0, total_ev1

    total0 = total1 = 0.0
    for c0, c1 in deals:
        e0, e1 = ev_for_deal(pkpy.KuhnState(c0, c1))
        total0 += e0
        total1 += e1

    # Divide by number of deals for expected value per hand
    n = len(deals)
    return total0 / n, total1 / n


# Compare default strategy vs GTO at multiple alpha values
default_strategy = pkpy.KuhnStrategy.default()
gto_ev = compute_kuhn_ev(pkpy.KuhnStrategy.gto(1.0 / 3.0))
default_ev = compute_kuhn_ev(default_strategy)

nash_ev = -1.0 / 18.0  # exact analytical Nash equilibrium EV for P0

print("Kuhn Poker EV per hand (ante = 1 chip)")
print(f"{'Strategy':<28} {'P0 EV':>10} {'P1 EV':>10}")
print("-" * 52)
print(f"{'Default (heuristic)':<28} {default_ev[0]:>+10.4f} {default_ev[1]:>+10.4f}")
print(f"{'GTO (alpha=1/3)':<28} {gto_ev[0]:>+10.4f} {gto_ev[1]:>+10.4f}")
print(f"{'Nash equilibrium (exact)':<28} {nash_ev:>+10.4f} {-nash_ev:>+10.4f}")

### Quiz 6 — EV Under Different Strategies

**Q1:** Player 0's Nash equilibrium EV is −1/18 ≈ −0.056 chips/hand. What structural feature of Kuhn Poker causes this built-in disadvantage?

**Q2:** The "default" strategy (always bet King, check everything else) — is its EV better or worse than Nash for P0? Why?

**Q3:** Does the GTO strategy *maximise* P0's EV, or does it *minimise* P1's ability to exploit P0? What's the difference?

> Run the answer cell below to check.

In [ ]:
# Quiz 6 — Answer Reveal

print("Q1  P0 acts FIRST without seeing P1's card.")
print("    Acting out of position leaks information — your action (bet/check) is observable")
print("    before P1 commits. P1 always has more information when they act.")
print("    This positional disadvantage is worth exactly 1/18 chips/hand analytically.")
print()

# Q2: default vs Nash EV comparison
default_p0_ev, _ = compute_kuhn_ev(pkpy.KuhnStrategy.default())
nash_exact = -1.0 / 18.0
print(f"Q2  Default (always bet K, check rest) P0 EV: {default_p0_ev:+.6f}")
print(f"    Nash equilibrium P0 EV:                  {nash_exact:+.6f}")
worse_better = "worse" if default_p0_ev < nash_exact else "better"
print(f"    Default is {worse_better} than Nash. A predictable 'only value-bet' strategy")
print(f"    lets a smart P1 always fold to bets → P0 never wins with bluffs.")
print()

print("Q3  GTO does NOT maximise EV against all opponents.")
print("    It MINIMISES exploitability — guaranteeing the Nash floor (−1/18 for P0).")
print("    Against an exploitable opponent, a custom exploitative strategy earns MORE.")
print("    GTO is a defensive guarantee, not an income maximiser.")

## 7. Exploitability: How Much EV Does a Non-GTO Strategy Leave?

**Exploitability** measures how much EV you give up relative to Nash equilibrium. A strategy with high exploitability can be defeated by an opponent who adjusts to exploit your tendencies.

The `KuhnCfr` class implements **Counterfactual Regret Minimization (CFR)** — an algorithm that iteratively converges to the Nash equilibrium by minimizing regret over many training iterations.

Let's visualise how exploitability shrinks as training progresses.

In [ ]:
# Train CFR and track exploitability at checkpoints
cfr = pkpy.KuhnCfr()

checkpoints = [0, 10, 50, 100, 500, 1_000, 5_000, 10_000, 50_000]
results = []

prev = 0
for checkpoint in checkpoints:
    additional = checkpoint - prev
    if additional > 0:
        cfr.train(additional)
    exploit = cfr.exploitability()
    results.append((checkpoint, exploit))
    prev = checkpoint

print(f"{'Iterations':>12} {'Exploitability':>16} {'% of Initial':>14}")
print("-" * 46)
initial_exploit = results[0][1]
for iters, exploit in results:
    pct = (exploit / initial_exploit) * 100
    bar = "█" * int((exploit / initial_exploit) * 30)
    print(f"{iters:>12,} {exploit:>16.6f} {pct:>13.1f}%  {bar}")

### Quiz 7 — Exploitability & CFR

**Q1:** At 0 training iterations, CFR has done nothing — what is the exploitability and why is it high?

**Q2:** If exploitability = 0.05 chips/hand after training, what does that mean in practical terms for a 10,000-hand session?

**Q3:** The convergence plot shows exploitability generally decreasing — but it doesn't always decrease monotonically. Why might it spike upward at certain checkpoints?

> Run the answer cell below to check.

In [ ]:
# Quiz 7 — Answer Reveal

print(f"Q1  At 0 iterations, CFR uses uniform random probabilities (50/50 check/bet).")
print(f"    Exploitability at 0 iterations: {results[0][1]:.6f} chips/hand")
print(f"    A best-responding opponent extracts that much extra EV per hand over Nash.")
print()

print("Q2  Exploitability = 0.05 chips/hand means:")
print("    A fully best-responding opponent earns 0.05 chips/hand MORE than Nash baseline.")
print("    Over 10,000 hands that's 500 chips. In any significant game, highly costly.")
print()

print("Q3  CFR tracks *cumulative average* strategy, not the current policy.")
print("    The *current* iterate fluctuates — it's the average that converges.")
print("    Checkpoints may land at high-variance current iterates, not the average,")
print("    so exploitability can appear to spike at certain points while trending down.")
print()
print("Exploitability convergence:")
for iters, exploit in results:
    bar = "█" * max(1, int((exploit / results[0][1]) * 30))
    print(f"  {iters:>8,} iters: {exploit:.6f}  {bar}")

## 8. GTO Action Probabilities at Nash Equilibrium

After training, let's examine what action probabilities the CFR algorithm converged to — and compare them to the known closed-form GTO solution.

**Kuhn Poker Nash equilibrium** (for alpha = 1/3):

| Player | Card | Situation      | Bet/Call prob |
|--------|------|----------------|---------------|
| P0     | J    | Open           | 1/3           |
| P0     | Q    | Open           | 0             |
| P0     | K    | Open           | 1             |
| P1     | J    | Facing check   | 0             |
| P1     | Q    | Facing check   | 1/3           |
| P1     | K    | Facing check   | 1             |
| P1     | J    | Facing bet     | 0             |
| P1     | Q    | Facing bet     | 1/3           |
| P1     | K    | Facing bet     | 1             |
| P0     | J    | After check-bet | 0            |
| P0     | Q    | After check-bet | 0            |
| P0     | K    | After check-bet | 1            |

In [ ]:
def describe_action_probs(strategy: pkpy.KuhnStrategy, label: str) -> None:
    """
    Print action probabilities for every reachable information set.

    Args:
        strategy: A KuhnStrategy to inspect.
        label:    Display name for this strategy.
    """
    cards = [pkpy.KuhnCard.JACK, pkpy.KuhnCard.QUEEN, pkpy.KuhnCard.KING]

    print(f"\nStrategy: {label}")
    print(f"{'Info Set':<18} {'Actions & Probabilities'}")
    print("-" * 60)

    seen = set()

    def inspect(state: pkpy.KuhnState):
        if state.is_terminal():
            return
        current = state.current_player()
        info_set = state.info_set(current)
        key = str(info_set)
        if key not in seen:
            seen.add(key)
            probs = strategy.action_probs(info_set)
            prob_str = "  ".join(f"{str(a)}:{p:.3f}" for a, p in probs)
            print(f"P{current} {key:<16} {prob_str}")
        for action in state.legal_actions():
            inspect(state.apply(action))


    for c0 in cards:
        for c1 in cards:
            if c0 != c1:
                inspect(pkpy.KuhnState(c0, c1))


# Compare GTO analytical vs CFR-trained
gto_strategy = pkpy.KuhnStrategy.gto(1.0 / 3.0)
cfr_strategy = cfr.average_strategy()

describe_action_probs(gto_strategy, "GTO Analytical (alpha=1/3)")
describe_action_probs(cfr_strategy, f"CFR Trained ({checkpoints[-1]:,} iterations)")

### Quiz 8 — GTO Action Probabilities

Study the action probability tables printed above, then answer:

**Q1:** In the Nash equilibrium (alpha=1/3), P0 bluffs with the Jack at 33%. Why not 0% or 100%?

**Q2:** P1 holds the Queen and faces a P0 bet. The Nash solution says P1 calls at 1/3 frequency. What keeps this from being 0% or 100%?

**Q3 (Fill in the blank):** If P0 *never* bluffs with the Jack, a smart P1 should _____ whenever P0 bets, because P0 can only be holding the _____.

> Run the answer cell below to check.

In [ ]:
# Quiz 8 — Answer Reveal

print("Q1  P0 bluffs with J at alpha = 1/3 (≈ 33%).")
print("    • 0% bluffing  → P1 always folds to bets (only K bets) → P0 loses EV")
print("    • 100% bluffing → P1 always calls → bluffing with J is always -EV")
print("    • 1/3 is the exact frequency that makes P1 *indifferent* to calling with Q.")
print("    This is the mixed-strategy Nash equilibrium: no unilateral deviation is profitable.")
print()

print("Q2  P1 with Q calls a P0 bet at frequency 1/3.")
print("    P0 bets with: K (always) + J (alpha = 1/3 of the time).")
print("    P1 calling at 1/3 makes P0 exactly indifferent between bluffing and checking with J.")
print("    ↑ call more → bluffing with J becomes -EV (bad for P0) → P0 bluffs less")
print("    ↓ call less → bluffing with J becomes +EV (good for P0) → P0 bluffs more")
print("    At 1/3 neither player has an incentive to deviate.")
print()

print("Q3  If P0 never bluffs:")
print("    P1 should ALWAYS FOLD when P0 bets, because P0 can only be betting the KING.")
print("    Result: P0's King always gets folds, never extracts value from bets.")
print("    Bluffing is necessary to give your value bets credibility.")

## 9. EV Comparison: Different Strategies vs GTO

Finally, let's put it all together. We compare the EV of:

- A **pure strategy** (always bet the King, always check everything else)
- An **exploitable bluffing strategy** (always bluff with the Jack)
- The **GTO strategy** at different alpha values
- The **CFR-trained strategy**

Any deviation from the Nash equilibrium can be **exploited** by a best-response opponent — they will extract more EV against you than the Nash baseline of +1/18 per hand for P1.

In [ ]:
alpha_values = [0.0, 1.0 / 6.0, 1.0 / 3.0, 1.0 / 2.0]

print("P0 EV per hand under GTO strategies with different alpha values")
print("(alpha = P0's Jack bluffing frequency; Nash-optimal range: [0, 1/3])")
print()
print(f"{'alpha':>10} {'P0 EV':>12} {'P1 EV':>12} {'Exploitability':>16}")
print("-" * 55)

for alpha in alpha_values:
    strat = pkpy.KuhnStrategy.gto(alpha)
    e0, e1 = compute_kuhn_ev(strat)
    exploit = cfr.exploitability()  # Use cfr as reference; strategy-specific would need separate cfr
    # Compute per-strategy exploitability by checking deviation from Nash value
    print(f"{alpha:>10.4f} {e0:>+12.6f} {e1:>+12.6f}")

print()
print(f"Nash exact P0 EV: {-1/18:>+.6f}  (= -1/18)")
print()

# Summary
cfr_e0, cfr_e1 = compute_kuhn_ev(cfr_strategy)
print(f"CFR-trained EV after {checkpoints[-1]:,} iterations:")
print(f"  P0: {cfr_e0:+.6f}")
print(f"  P1: {cfr_e1:+.6f}")
print(f"  Exploitability: {cfr.exploitability():.6f} chips/hand")

### Quiz 9 — EV Across Strategies

**Q1:** The Nash-optimal alpha range for P0's Jack bluffing is [0, 1/3]. What happens to P0's EV if alpha = 1/2 (P0 over-bluffs)?

**Q2:** A player argues: "I should always bluff the maximum — more bluffs means more fold equity." Based on everything in this notebook, what's wrong with this reasoning?

**Q3:** The CFR algorithm converges to Nash equilibrium. But in a real poker game you're not playing a Nash bot — when is it *better* to exploit your opponent than play GTO?

> Run the answer cell below to check.

In [ ]:
# Quiz 9 — Answer Reveal

# Q1: over-bluffing EV impact
ev_nash_strat = compute_kuhn_ev(pkpy.KuhnStrategy.gto(1.0 / 3.0))
ev_overbluff = compute_kuhn_ev(pkpy.KuhnStrategy.gto(0.5))

print(f"Q1  P0 EV at alpha=1/3 (Nash):        {ev_nash_strat[0]:+.6f}   P1: {ev_nash_strat[1]:+.6f}")
print(f"    P0 EV at alpha=1/2 (over-bluff):  {ev_overbluff[0]:+.6f}   P1: {ev_overbluff[1]:+.6f}")
p1_gain = ev_overbluff[1] - ev_nash_strat[1]
print(f"    Over-bluffing costs P0 {abs(ev_overbluff[0] - ev_nash_strat[0]):.6f} chips/hand vs Nash.")
print(f"    P1 gains {p1_gain:+.6f} chips/hand by exploiting the over-bluff.")
print()

print("Q2  'Always bluff maximum' is flawed because:")
print("    • A rational opponent adjusts: they call more, turning your bluffs into losses.")
print("    • Optimal bluff frequency balances your range so villain is *indifferent*.")
print("    • In Kuhn: alpha > 1/3 → P1 calling with Q becomes +EV, exploiting P0.")
print("    • More bluffs is only better if villain *doesn't adjust* — a dangerous assumption.")
print()

print("Q3  Exploit opponents WHEN:")
print("    • You have a reliable read (they over-fold to bets, always call rivers, etc.)")
print("    • The session is short — exploitation wins faster than the Nash guarantee")
print("    • Opponent is unlikely to adjust (recreational players, one-off sessions)")
print()
print("    Play GTO WHEN:")
print("    • Opponent is unknown or highly adaptive — GTO guarantees the Nash floor")
print("    • You want an unexploitable baseline in tough lineups")
print("    • Learning environments — GTO trains correct range-balancing instincts")

## Summary

| Concept | Key Takeaway |
|---------|-------------|
| **EV formula** | `EV = Σ P(outcome) × value(outcome)` — use it for every decision |
| **Pot odds** | Break-even equity = `call / (pot + 2×call)` — know this cold |
| **Range equity** | `pkpy.Versus` enumerates blockers and computes exact equity vs a range |
| **Outs counting** | `pkpy.Game.turn_case_evals()` gives exact river-card outcomes |
| **Kuhn game tree** | `KuhnState.apply()` + `is_terminal()` + `payoff()` = full CFR traversal |
| **Nash equilibrium** | GTO minimises exploitability; CFR converges to it iteratively |
| **Bluffing is math** | In Kuhn Poker, optimal bluff frequency is derived analytically — not a feeling |